# Experiment: Case-001 No-Lab Completion Tracker

Objective:
- Convert the no-lab execution ladder into a public-safe completion tracker.
- Keep private record handling separate from public case-context writing.
- Rank missing domains without using raw patient records.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class Domain:
    """One public-safe case-packet completion domain."""

    name: str
    current_label: str
    evidence_weight: int
    referral_weight: int
    privacy_risk: int
    public_ready: bool

    @property
    def priority(self) -> int:
        """Return higher priority for missing, clinically useful domains."""
        readiness_penalty = 0 if self.public_ready else 3
        return (
            self.evidence_weight
            + self.referral_weight
            + readiness_penalty
            - self.privacy_risk
        )

## Public-Safe Domain Inputs

These are labels from tracked case-context files, not raw medical records.


In [ ]:
domains = [
    Domain("diagnosis_and_genotype", "phenotype_only", 5, 5, 2, False),
    Domain(
        "transfusion_burden",
        "transfusion_dependent_burden_unquantified",
        5,
        5,
        2,
        False,
    ),
    Domain("immune_blood_bank", "immune_transfusion_packet_missing", 5, 5, 2, False),
    Domain("iron_chelation_organ_risk", "iron_packet_missing", 5, 5, 2, False),
    Domain(
        "advanced_referral", "advanced_therapy_referral_packet_missing", 4, 5, 1, False
    ),
    Domain("private_intake", "private_intake_index_needed", 5, 3, 3, False),
    Domain("public_release", "release_check_required", 4, 4, 2, False),
]

ranked_domains = sorted(
    (
        {
            "domain": domain.name,
            "label": domain.current_label,
            "priority": domain.priority,
            "public_ready": domain.public_ready,
        }
        for domain in domains
    ),
    key=lambda item: item["priority"],
    reverse=True,
)
ranked_domains

## Gate Decision

No domain is public-ready until the private source index and release checklist are complete. The tracker should remain a checklist and clinician-question artifact, not a treatment plan.


In [ ]:
decision = {
    "tracker_label": "case001_no_lab_completion_tracker_active",
    "top_blockers": [item["domain"] for item in ranked_domains[:4]],
    "public_case_update_allowed": False,
    "patient_specific_claims": "blocked_until_clinician_review",
    "next_public_safe_artifact": "case_001_no_lab_completion_tracker",
}
decision